# Entropy-Gated Amnesic — Can we break the +2.7pp Pareto ceiling?

**Key discovery from prefill sweep**: `L17_last_half` gives **+3.6pp** but breaks Pareto with 3 losses. `every_4` gives +3.6pp with 1 loss. There IS signal for stronger ablation, but it comes with noise.

**Hypothesis (MTI-inspired, arXiv:2510.13940)**: the losses from broader ablation occur on **low-entropy items** (model confident, ablation hurts). The gains occur on **high-entropy items** (model uncertain, ablation helps). An entropy-gated policy — apply broader ablation only when model is uncertain — should recover gains without losses.

MTI achieved **+9.28pp average** on reasoning via entropy-gated classifier-free guidance. We test the analogous gating for amnesic.

**Protocol** (standalone, ~15 min):
1. Run 4 variants per item: baseline + L17_last + L17_last_half + L17_every_4
2. Save baseline entropy over letter logits for each item
3. Offline analysis: try multiple entropy thresholds × ablation variants → find best gated policy

**Success criteria**:
- Any gated policy with **Δ > +3pp AND lost = 0** → **first breach of the +2.7pp Pareto ceiling**
- Near-zero-loss variant at +4-5pp → publication-worthy positive

## Cell 1 — Install

In [ ]:
import sys, subprocess, os, shutil
from pathlib import Path

def pip(*a, check=True):
    return subprocess.run([sys.executable, '-m', 'pip', *a], check=check)

try:
    import transformers
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    ok = 'qwen3_5_moe' in CONFIG_MAPPING_NAMES
except Exception:
    ok = False

if not ok:
    pip('install', '-q', 'accelerate', 'datasets', 'huggingface_hub==1.5.0',
        'safetensors', 'einops', 'scikit-learn', 'sentencepiece', 'tokenizers',
        'protobuf', 'scipy', 'matplotlib', 'hf_transfer', check=False)
    pip('uninstall', '-y', 'transformers', check=False)
    SRC = '/content/transformers_src'
    if Path(SRC).exists(): shutil.rmtree(SRC)
    subprocess.run(['git','clone','--quiet','--depth=1',
                    'https://github.com/huggingface/transformers.git', SRC], check=True)
    pip('install','--force-reinstall','--no-deps','--no-cache-dir', SRC, check=False)
    pip('install','--no-cache-dir','flash-linear-attention', check=False)
    pip('install','-q','--no-cache-dir',
        'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1.post4/'
        'causal_conv1d-1.6.1%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl',
        check=False)
    for m in list(sys.modules):
        if m.startswith('transformers') or m.startswith('huggingface_hub'):
            del sys.modules[m]

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        from huggingface_hub import login
        login(token=hf_token)
        print('HF auth OK')
except Exception as e:
    print(f'(skipping colab auth: {e})')

import json, re, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

OUT = Path('/content/entropy_gated'); OUT.mkdir(exist_ok=True)
print('setup done')

## Cell 2 — Load model

In [ ]:
from transformers import AutoTokenizer, AutoModelForImageTextToText
from huggingface_hub import snapshot_download
from safetensors import safe_open

MODEL_ID = 'Qwen/Qwen3.6-35B-A3B'
LAYERS = [17]

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token_id is None: tok.pad_token = tok.eos_token

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map='cuda',
    attn_implementation='sdpa', trust_remote_code=True)
model.eval()
print(f'Model loaded: {torch.cuda.memory_allocated()/1e9:.1f} GB')

## Cell 3 — Fit probe L17 + build letter directions

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

corpus = snapshot_download('caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b',
                            repo_type='dataset', cache_dir='/content/cache')
shards = sorted((Path(corpus)/'shards').glob('q*.safetensors'))

MIN_LEN = 200; POOL_WINDOW = 10
pooled, labels = [], []
for shard in tqdm(shards, desc='collect'):
    with safe_open(str(shard), framework='pt') as f:
        meta = f.metadata()
        offs = json.loads(meta['offsets'])['L17']
        rollouts = json.loads(meta['rollouts'])
        acts = f.get_tensor('acts_L17')
        for r_idx, r in enumerate(rollouts):
            if r['pred'] is None or r['response_len'] < MIN_LEN: continue
            ra = acts[offs[r_idx]:offs[r_idx+1]].float().numpy()
            pooled.append(ra[:POOL_WINDOW].mean(axis=0))
            labels.append(r['pred'])

pooled = np.stack(pooled); labels = np.array(labels)
letters = sorted(set(labels))
letter_to_int = {l: i for i, l in enumerate(letters)}
y = np.array([letter_to_int[l] for l in labels])

scaler = StandardScaler().fit(pooled)
pca = PCA(n_components=128, random_state=42).fit(scaler.transform(pooled))
logreg = LogisticRegression(C=1.0, max_iter=3000, random_state=42).fit(
    pca.transform(scaler.transform(pooled)), y)
print(f'L17 train: {logreg.score(pca.transform(scaler.transform(pooled)), y):.3f}')

dirs = []
for li in range(len(letters)):
    coef = logreg.coef_[li]
    d_scaled = pca.components_.T @ coef
    d_raw = d_scaled * scaler.scale_
    d_raw = d_raw / (np.linalg.norm(d_raw) + 1e-8)
    dirs.append(torch.from_numpy(d_raw.astype(np.float32)).cuda().to(torch.bfloat16))
d_stack = torch.stack(dirs)  # [10, d_model]
print(f'✅ L17 d_stack {d_stack.shape}')

## Cell 4 — Prefill-aware amnesic hook

In [ ]:
def get_layer_module(m, idx):
    cands = [m]
    if hasattr(m, 'model'): cands.append(m.model)
    for start in cands:
        for path in [('model','language_model','layers'),('language_model','layers'),('model','layers'),('layers',)]:
            cur = start; ok = True
            for p in path:
                if hasattr(cur, p): cur = getattr(cur, p)
                else: ok = False; break
            if ok and hasattr(cur, '__getitem__'):
                try: return cur[idx]
                except: continue
    raise RuntimeError(f'layer {idx} not found')

class AmnesicHookFlex:
    """Apply L17 amnesic at configurable position set during prefill."""
    def __init__(self):
        self.mode = 'off'
        self.positions_mode = 'last'  # 'last' | 'last_half' | 'every_N'
        self.prompt_len = None
        self.alpha = 1.0
        self.every_n = 1

    def set(self, positions_mode, prompt_len, alpha=1.0, every_n=1):
        self.mode = 'on'
        self.positions_mode = positions_mode
        self.prompt_len = prompt_len
        self.alpha = alpha
        self.every_n = every_n

    def off(self):
        self.mode = 'off'

    def make_hook(self):
        def hook(module, inp, out):
            if self.mode == 'off': return out
            hidden = out[0] if isinstance(out, tuple) else out
            T = hidden.shape[1]
            if T != self.prompt_len: return out
            if self.positions_mode == 'last':
                positions = [T - 1]
            elif self.positions_mode == 'last_half':
                positions = list(range(T // 2, T))
            elif self.positions_mode == 'every_N':
                positions = list(range(0, T, self.every_n))
            else:
                return out
            hidden = hidden.clone()
            x_all = hidden[0]
            coefs = x_all[positions] @ d_stack.T
            delta = coefs @ d_stack
            hidden[0, positions, :] = x_all[positions] - self.alpha * delta
            if isinstance(out, tuple): return (hidden,) + out[1:]
            return hidden
        return hook

amnesic = AmnesicHookFlex()
h_ab = get_layer_module(model, 17).register_forward_hook(amnesic.make_hook())
print('✅ flex amnesic hook on L17')

## Cell 5 — Load eval questions

In [ ]:
questions = []
for shard in shards:
    with safe_open(str(shard), framework='pt') as f:
        meta = f.metadata()
        rollouts_ = json.loads(meta['rollouts'])
        opts = json.loads(meta['options'])
        n_correct = sum(1 for r in rollouts_ if r['correct'])
        if len(opts) == 10 and n_correct >= 1:
            questions.append(dict(
                question=meta['question'], options=opts,
                gold_letter=meta['gold_letter'], n_options=10, n_correct=n_correct))
print(f'{len(questions)} Stage B 10-opt questions')

def format_mcq(question, options):
    choices = '\n'.join(f'{chr(65+i)}. {opt}' for i, opt in enumerate(options))
    content = ("Answer the following multiple-choice question. "
        "Give ONLY the letter of the correct answer.\n\n"
        f"Question: {question}\n\nOptions:\n{choices}\n\nAnswer: \\boxed{{")
    return tok.apply_chat_template([{'role':'user','content':content}],
                                    tokenize=False, add_generation_prompt=True,
                                    enable_thinking=False)

letter_ids = {L: tok(L, add_special_tokens=False).input_ids[0] for L in 'ABCDEFGHIJ'}

## Cell 6 — Sweep: baseline + 3 amnesic variants, save entropies

For each item, compute:
- `baseline` (no ablation) + entropy of letter logits
- `amn_last` (L17 α=1.0 last token)
- `amn_last_half` (L17 α=1.0 last half prefill)
- `amn_every4` (L17 α=1.0 every 4 tokens prefill)

In [ ]:
random.seed(42)
sample = random.sample(questions, min(200, len(questions)))
print(f'{len(sample)} items × 4 configs')

VARIANTS = [
    ('baseline',     None,      1.0, 1),
    ('amn_last',     'last',    1.0, 1),
    ('amn_last_half','last_half', 1.0, 1),
    ('amn_every4',   'every_N', 1.0, 4),
]

def run_variant(ids, prompt_len, pos_mode, alpha, every_n, n_opts):
    if pos_mode is None:
        amnesic.off()
    else:
        amnesic.set(pos_mode, prompt_len, alpha, every_n)
    with torch.no_grad():
        out = model(input_ids=ids, attention_mask=torch.ones_like(ids))
    logits = out.logits[0, -1]
    letter_logits = torch.tensor([logits[letter_ids[L]].item() for L in 'ABCDEFGHIJ'[:n_opts]])
    probs = F.softmax(letter_logits, dim=0)
    entropy = -(probs * (probs.clamp(min=1e-12)).log()).sum().item()
    pred_idx = int(letter_logits.argmax().item())
    pred = 'ABCDEFGHIJ'[pred_idx]
    return pred, entropy, letter_logits.tolist()

results = []
t0 = time.time()
for i, q in enumerate(tqdm(sample, desc='entropy sweep')):
    try:
        p = format_mcq(q['question'], q['options'])
        ids = tok(p, return_tensors='pt').input_ids.to('cuda')
        if ids.shape[1] > 4096: continue
        prompt_len = ids.shape[1]
        row = dict(idx=i, gold=q['gold_letter'], n_options=q['n_options'], prompt_len=prompt_len)
        for (name, pos_mode, alpha, every_n) in VARIANTS:
            pred, entropy, letter_logits = run_variant(ids, prompt_len, pos_mode, alpha, every_n, q['n_options'])
            row[f'{name}_pred'] = pred
            row[f'{name}_correct'] = (pred == q['gold_letter'])
            row[f'{name}_entropy'] = entropy
            row[f'{name}_logits'] = letter_logits
        amnesic.off()
        results.append(row)
        if (i+1) % 20 == 0:
            accs = {v[0]: sum(r[f'{v[0]}_correct'] for r in results)/len(results) for v in VARIANTS}
            print(f'  [{i+1}/{len(sample)}] ' + ' | '.join(f'{n}={a*100:.0f}%' for n,a in accs.items()))
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); continue

with open(OUT/'entropy_results.json', 'w') as f:
    json.dump(dict(n=len(results), results=results, wall_min=(time.time()-t0)/60), f, indent=2)
print(f'\n✅ {(time.time()-t0)/60:.1f} min')

## Cell 7 — Entropy-gated policy search

In [ ]:
from scipy.stats import binomtest

# Distribution of entropies
entropies = np.array([r['baseline_entropy'] for r in results])
print(f'Entropy stats: min={entropies.min():.3f} | median={np.median(entropies):.3f} | mean={entropies.mean():.3f} | max={entropies.max():.3f}')
print(f'Quantiles: 25%={np.quantile(entropies,0.25):.3f} | 50%={np.quantile(entropies,0.5):.3f} | 75%={np.quantile(entropies,0.75):.3f}')

# For each (variant, threshold) combination, apply gated policy and measure
THRESHOLDS = [0.0, 0.3, 0.5, 0.8, 1.0, 1.3, 1.5, 1.8, 2.0]
VARIANT_NAMES = ['amn_last', 'amn_last_half', 'amn_every4']

print(f'\n=== Gated policy: use variant if entropy > threshold, else baseline ===')
print(f'{"variant":15s} {"thresh":>7s}  {"acc%":>6s}  {"Δpp":>6s}  {"gain":>5s}  {"lost":>5s}  {"p":>7s}')
baseline_correct = [r['baseline_correct'] for r in results]
base_acc = sum(baseline_correct) / len(results) * 100
print(f'baseline        :        : {base_acc:5.1f}%\n')

all_stats = []
for variant in VARIANT_NAMES:
    for thresh in THRESHOLDS:
        gated = []
        for r in results:
            if r['baseline_entropy'] > thresh:
                gated.append(r[f'{variant}_correct'])  # use amnesic
            else:
                gated.append(r['baseline_correct'])  # use baseline
        acc = sum(gated) / len(gated)
        delta = (acc - base_acc/100) * 100
        g = sum(1 for bc, gc in zip(baseline_correct, gated) if not bc and gc)
        l = sum(1 for bc, gc in zip(baseline_correct, gated) if bc and not gc)
        p = binomtest(g, g+l, p=0.5, alternative='two-sided').pvalue if g+l > 0 else 1.0
        trigger_rate = (entropies > thresh).mean()
        all_stats.append((variant, thresh, acc*100, delta, g, l, p, trigger_rate))

all_stats.sort(key=lambda r: (-r[3], r[5]))  # sort by Δ desc, then lost asc
for (variant, thresh, acc, delta, g, l, p, tr) in all_stats:
    if delta > 5 and l == 0: star = ' ***'
    elif delta > 3 and l == 0: star = ' **'
    elif delta > 1.5 and l == 0: star = ' *'
    elif delta > 0.5 and l == 0: star = ' +'
    else: star = ''
    print(f'{variant:15s} thr={thresh:>3.1f}  {acc:5.1f}%  {delta:+5.1f}  {g:>5d}  {l:>5d}  {p:>7.3f}  (trig {tr*100:.0f}%){star}')

# Best overall + best Pareto
best = max(all_stats, key=lambda r: r[3])
best_pareto = max([r for r in all_stats if r[5] == 0], key=lambda r: r[3], default=None)
print('\n=== VERDICT ===')
if best_pareto and best_pareto[3] > 3:
    print(f'*** BREACH: {best_pareto[0]} @ thresh={best_pareto[1]} gives {best_pareto[3]:+.1f}pp Pareto (0 lost)')
    print(f'    Triggers on {best_pareto[7]*100:.0f}% of items.')
    print(f'    BEATS +2.7pp ceiling from non-gated amnesic.')
elif best_pareto and best_pareto[3] > 1.5:
    print(f'*  Moderate: {best_pareto[0]} @ thresh={best_pareto[1]} gives {best_pareto[3]:+.1f}pp Pareto')
    print(f'   Not a breach, but confirms gating works.')
else:
    print(f'-- No Pareto breach: best gated {best[3]:+.1f}pp with {best[5]} lost')
    print(f'   Entropy gating does not rescue the losses from broader ablation.')

print(f'\n--- Reference (non-gated) ---')
for v in VARIANT_NAMES:
    cc = [r[f'{v}_correct'] for r in results]
    acc = sum(cc)/len(results)
    g = sum(1 for bc,c in zip(baseline_correct, cc) if not bc and c)
    l = sum(1 for bc,c in zip(baseline_correct, cc) if bc and not c)
    print(f'  {v:15s}: {acc*100:5.1f}%  Δ{(acc-base_acc/100)*100:+.1f}pp  g={g} l={l}')